# Лабораторная работа № 4. Генерация синтетических данных с помощью GAN

Цель: объяснить роли G и D, реализовать ключевые части их обучения, получить новые данные,
управлять классом в cGAN и оценить сходство **и разнообразие** результата.
Градиенты MLP уже знакомы по предыдущим работам, поэтому инфраструктура предоставлена.

Работайте сверху вниз, заполняйте TODO и Markdown-поля «Ответ». Остановки
`NotImplementedError` намеренны. Для сдачи перезапустите ядро, выполните все ячейки
и сохраните заполненный `.ipynb` с выводами. Отдельный отчет не требуется.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter

SEED=41
Z_DIM=2
BATCH=128
STEPS=2200       # После первого прогона можно исследовать другие значения отдельно.
np.set_printoptions(precision=4,suppress=True)
print('NumPy:',np.__version__)


## A. Что генерирует сеть и что оценивает дискриминатор

Учебные «реальные» данные тоже смоделированы: $y=0$ соответствует
$x\sim\mathcal N(-1.5,0.25^2)$, а $y=1$ — $x\sim\mathcal N(1.5,0.35^2)$.
В обучающем наборе 540 и 60 примеров, то есть 90% и 10%.
Здесь «реальные» означает опорные данные для D, а не реальные измерения предметной области.
Для оценки заранее создается независимый набор с тем же соотношением классов.

Обычная GAN использует две модели:
$$z\sim\mathcal N(0,I_2),\quad \tilde x=G(z;\theta_G),\quad
D(x;\theta_D)=\sigma(a(x;\theta_D)).$$
G: **2 → 16 → 1**, D: **1 → 16 → 1**. В скрытом слое tanh;
выход G линейный, D возвращает логит a, а вероятность вычисляем сигмоидой отдельно.
Две сети — выбор учебной реализации; термин GAN задает принцип обучения, не запрет иных архитектур.

**1.** Допишите выборку стандартного нормального шума формы (n,2).

**2.** Укажите входы/выходы обеих моделей. Объясните, где находится латентное пространство.

Начальное обучение меняет веса G и D; обычно z каждый раз выбирается заново.
Подбор латентного кода при фиксированном G — другая задача.


In [ ]:
def draw_noise(rng,n):
    # TODO: n независимых двумерных стандартных нормальных векторов
    raise NotImplementedError('Допишите draw_noise')

rng_data=np.random.default_rng(SEED)
y_train=np.repeat([0,1],[540,60]).reshape(-1,1)
x_train=(np.where(y_train==0,-1.5,1.5)
         +np.where(y_train==0,.25,.35)*rng_data.normal(size=y_train.shape))
rng_eval=np.random.default_rng(901)
y_eval=np.repeat([0,1],[1800,200]).reshape(-1,1)
x_eval=(np.where(y_eval==0,-1.5,1.5)
        +np.where(y_eval==0,.25,.35)*rng_eval.normal(size=y_eval.shape))
assert draw_noise(np.random.default_rng(1),7).shape==(7,Z_DIM)
print('Примеров обучения:',len(x_train),'по классам:',np.bincount(y_train.ravel()))
plt.hist(x_train.ravel(),bins=50,density=True,alpha=.6,label='Обучающие опорные данные')
plt.xlabel('x');plt.ylabel('Плотность');plt.legend();plt.grid();plt.show()


**Ответ A.** Моделей … . G получает … и выдает …; D получает … и выдает … .
Шум отличается от обучающих примеров тем, что … . Почему G не обязан буквально смешивать строки x_train? … .


## B. Состязательные функции потерь

В лекции дана минимаксная игра:
$$\min_G\max_D\ \mathbb E_x\log D(x)+\mathbb E_z\log(1-D(G(z))).$$
В коде **минимизируются** обе функции, но по разным весам:
$$L_D=\operatorname{mean}(\operatorname{softplus}(-a_{real}))+
\operatorname{mean}(\operatorname{softplus}(a_{fake})),$$
$$L_G=\operatorname{mean}(\operatorname{softplus}(-a_{fake})).$$
Здесь $\operatorname{softplus}(u)=\log(1+e^u)$; устойчивое вычисление — `np.logaddexp(0,u)`.
**Для G используется non-saturating loss $-\log D(G(z))$**, а не буквально
минимаксный $\log(1-D(G(z)))$. Обе постановки стремятся сделать fake
убедительным для D, но их градиенты различаются. Такой прием описан в исходной статье GAN.

Для батчей по n строк:
$$\frac{\partial L_D}{\partial a_{real}}=\frac{\sigma(a_{real})-1}{n},\quad
\frac{\partial L_D}{\partial a_{fake}}=\frac{\sigma(a_{fake})}{n},\quad
\frac{\partial L_G}{\partial a_{fake}}=\frac{\sigma(a_{fake})-1}{n}.$$

Допишите loss.


In [ ]:
def sigmoid(a):
    return 1/(1+np.exp(-np.clip(a,-500,500)))

def losses(a_real,a_fake):
    # TODO: loss_D и non-saturating loss_G; вернуть два числа
    raise NotImplementedError('Допишите losses')


Допишите три производные и проверьте их.

In [ ]:
def discriminator_derivatives(a_real,a_fake):
    # TODO: две производные L_D по логитам реальных и синтетических примеров
    raise NotImplementedError('Допишите производные D')

def generator_derivative(a_fake):
    # TODO: производная non-saturating L_G по логиту D
    raise NotImplementedError('Допишите производную G')


In [ ]:
ar=np.array([[-.7],[.3]]);af=np.array([[.4],[-.8]])
dr,df=discriminator_derivatives(ar,af);dg=generator_derivative(af)
eps=1e-6
for i in range(2):
    h=np.zeros_like(ar);h[i]=eps
    assert np.isclose(dr[i,0],(losses(ar+h,af)[0]-losses(ar-h,af)[0])/(2*eps))
    assert np.isclose(df[i,0],(losses(ar,af+h)[0]-losses(ar,af-h)[0])/(2*eps))
    assert np.isclose(dg[i,0],(losses(ar,af+h)[1]-losses(ar,af-h)[1])/(2*eps))
ld0,lg0=losses(np.zeros((4,1)),np.zeros((4,1)))
assert np.allclose([ld0,lg0],[2*np.log(2),np.log(2)])
assert np.isfinite(losses(np.array([[-1000.]]),np.array([[1000.]]))).all()
print('Проверены loss и производные. При D=1/2: LD=',ld0,'LG=',lg0)


При идеальном равновесии распределения совпадают и оптимальный D дает 1/2,
но выход D около 1/2 сам по себе не доказывает качественную генерацию.


**Ответ B.** Какая метка у real/fake при обучении D? … . Какого ответа D добивается G? … .
Какие параметры меняет backprop на шаге G? … . Почему условие «D≈0.5» недостаточно? … .
Чем loss_G в коде отличается от формулы лекции? … .


## C. Попеременное обучение обычной GAN

Порядок одной итерации:
1. Получить real из обучающего набора и fake=G(z).
2. Вычислить градиенты L_D и обновить **только D**.
3. Выбрать новый z, провести fake через G и обновленный D.
4. Провести градиент L_G через D к G и обновить **только G**.

Готовая MLP возвращает выход и кэш, `backward` — градиенты параметров и входа.
Метод `backward` сам по себе веса не меняет. Готовый Adam меняет веса при `step`.

Впишите обновление D и три строки обновления G.


In [ ]:
class MLP:
    def __init__(self,n_in,h,n_out,rng):
        self.p=[rng.normal(0,1/np.sqrt(n_in),(n_in,h)),np.zeros(h),
                rng.normal(0,.2,(h,n_out)),np.zeros(n_out)]
    def forward(self,X):
        W,b,V,c=self.p
        H=np.tanh(X@W+b)
        return H@V+c,(X,H)
    def backward(self,grad,cache):
        X,H=cache;W,b,V,c=self.p
        hidden=(grad@V.T)*(1-H*H)
        return [X.T@hidden,hidden.sum(0),H.T@grad,grad.sum(0)],hidden@W.T

class Adam:
    def __init__(self,net,lr=.002):
        self.net=net;self.lr=lr;self.t=0
        self.m=[np.zeros_like(p) for p in net.p];self.v=[np.zeros_like(p) for p in net.p]
    def step(self,grads):
        self.t+=1
        for p,m,v,g in zip(self.net.p,self.m,self.v,grads):
            m[:]=.5*m+.5*g;v[:]=.999*v+.001*g*g
            p-=self.lr*(m/(1-.5**self.t))/(np.sqrt(v/(1-.999**self.t))+1e-8)

def with_condition(X,y=None):
    # Блок D переопределит эту функцию для условной генерации.
    if y is not None:raise ValueError('Сначала реализуйте with_condition в блоке D')
    return X


In [ ]:
def step_discriminator(G,D,opt_d,real,z,y=None):
    fake,_=G.forward(with_condition(z,y))
    ar,cache_r=D.forward(with_condition(real,y))
    af,cache_f=D.forward(with_condition(fake,y))
    dr,df=discriminator_derivatives(ar,af)
    grads_r,_=D.backward(dr,cache_r)
    grads_f,_=D.backward(df,cache_f)
    # TODO: сложить градиенты real/fake для каждого параметра и сделать шаг opt_d
    raise NotImplementedError('Допишите обновление D')

def step_generator(G,D,opt_g,z,y=None):
    fake,cache_g=G.forward(with_condition(z,y))
    af,cache_d=D.forward(with_condition(fake,y))
    _,grad_input_d=D.backward(generator_derivative(af),cache_d)
    # TODO: градиент по x (первый столбец), backward G, шаг opt_g; веса D не менять
    raise NotImplementedError('Допишите обновление G')


Запустите фиксированный эксперимент; сравните исходные и новые распределения.
Итерация здесь содержит два оптимизационных шага; это не полный проход по датасету.

In [ ]:
# Проверка: на каждом шаге меняются только параметры нужной сети.
probe_rng=np.random.default_rng(2)
Gp=MLP(2,4,1,probe_rng);Dp=MLP(1,4,1,probe_rng)
og=Adam(Gp);od=Adam(Dp);zp=draw_noise(probe_rng,8)
before_g=[p.copy() for p in Gp.p];before_d=[p.copy() for p in Dp.p]
step_discriminator(Gp,Dp,od,x_train[:8],zp)
assert all(np.array_equal(a,b) for a,b in zip(before_g,Gp.p))
assert any(not np.array_equal(a,b) for a,b in zip(before_d,Dp.p))
before_g=[p.copy() for p in Gp.p];before_d=[p.copy() for p in Dp.p]
step_generator(Gp,Dp,og,zp)
assert all(np.array_equal(a,b) for a,b in zip(before_d,Dp.p))
assert any(not np.array_equal(a,b) for a,b in zip(before_g,Gp.p))
print('Изоляция обновлений G/D проверена')

class_pool=[np.flatnonzero(y_train[:,0]==k) for k in [0,1]]
def train_gan(conditional=False,steps=STEPS,seed=SEED):
    rng=np.random.default_rng(seed)
    G=MLP(Z_DIM+int(conditional),16,1,rng)
    D=MLP(1+int(conditional),16,1,rng)
    og,od=Adam(G),Adam(D)
    z_fixed=draw_noise(np.random.default_rng(777),2000)
    labels_fixed=np.repeat([0,1],1000).reshape(-1,1) if conditional else None
    start=G.forward(with_condition(z_fixed,labels_fixed))[0]
    history=[]
    for it in range(steps):
        if conditional:
            c=rng.integers(0,2,(BATCH,1))
            # Обучение с балансировкой: одинаковый prior классов для real и fake.
            i0=rng.choice(class_pool[0],BATCH);i1=rng.choice(class_pool[1],BATCH)
            ids=np.where(c[:,0]==0,i0,i1)
        else:
            ids=rng.integers(0,len(x_train),BATCH);c=None
        real=x_train[ids]
        step_discriminator(G,D,od,real,draw_noise(rng,BATCH),c)
        step_generator(G,D,og,draw_noise(rng,BATCH),c)
        if it%100==0 or it==steps-1:
            # История — оба loss на одних текущих весах и диагностическом батче.
            fake,_=G.forward(with_condition(z_fixed[:BATCH],c))
            ar,_=D.forward(with_condition(real,c));af,_=D.forward(with_condition(fake,c))
            ld,lg=losses(ar,af)
            history.append((it+1,ld,lg,float(np.mean(sigmoid(ar))),float(np.mean(sigmoid(af)))))
    finish=G.forward(with_condition(z_fixed,labels_fixed))[0]
    return G,D,np.array(history),start,finish

t0=perf_counter()
G,D,history,fake_start,fake_unconditional=train_gan()
print('Обучение GAN, секунд:',round(perf_counter()-t0,2))
fig,axes=plt.subplots(1,2,figsize=(11,3))
for data,label in [(x_eval,'Независимый опорный набор'),(fake_start,'До обучения'),(fake_unconditional,'После обучения')]:
    axes[0].hist(data.ravel(),bins=60,range=(-3,3),density=True,histtype='step',label=label)
axes[0].set_xlabel('x');axes[0].set_ylabel('Плотность');axes[0].legend();axes[0].grid()
axes[1].plot(history[:,0],history[:,1],label='LD');axes[1].plot(history[:,0],history[:,2],label='LG')
axes[1].set_xlabel('Итерация');axes[1].set_ylabel('Loss');axes[1].legend();axes[1].grid()
plt.tight_layout();plt.show()
print('Доля x>0 у обычной GAN:',np.mean(fake_unconditional>0))


**Ответ C.** Как изменился результат? Сохранились ли оба режима распределения? … .
Почему уменьшение LG или рост D(fake) не гарантируют разнообразия? … .
Для получения новых строк после обучения нужны … . Нужно ли запускать D при каждом использовании G? … .
Что проверять до дополнения реального обучающего набора этими данными? … .


## D. cGAN и генерация заданного класса

$$\tilde x=G(z,y),\qquad D(x,y).$$
Здесь y — **условие класса**, а не real/fake метка. cGAN получает y в обе сети.
В нашем коде G: **3 → 16 → 1**, D: **2 → 16 → 1**; условие передается конкатенацией.

В опыте меняются и условные входы, и выборка обучающих классов: real/fake внутри батча
имеют одинаковый равномерный prior. Поэтому различия с базовой GAN нельзя приписывать
только конкатенации y. Данные независимой оценки не участвуют в обучении.
Это обычная учебная cGAN, **не SDG-GAN**: feature matching из SDG-GAN не реализован.

Допишите `with_condition`.

In [ ]:
def with_condition(X,y=None):
    if y is None:return X
    assert len(X)==len(y)
    # TODO: добавить столбец условия справа; вернуть новый массив
    raise NotImplementedError('Допишите условный вход')

assert with_condition(np.ones((3,2)),np.zeros((3,1))).shape==(3,3)
t0=perf_counter()
Gc,Dc,history_c,start_c,fake_c=train_gan(conditional=True)
print('Обучение cGAN, секунд:',round(perf_counter()-t0,2))


Запустите условное обучение. Выберите 1000 условий каждого класса и получите сбалансированную синтетику.

In [ ]:
# TODO: столбец условий с 1000 нулей и 1000 единиц, форма (2000,1)
y_synthetic=None
assert y_synthetic.shape==(2000,1)
assert np.array_equal(np.bincount(y_synthetic.ravel()),[1000,1000])
z_new=draw_noise(np.random.default_rng(2024),len(y_synthetic))
x_synthetic,_=Gc.forward(with_condition(z_new,y_synthetic))
fig,axes=plt.subplots(1,2,figsize=(10,3))
for k,ax in enumerate(axes):
    real=x_eval[y_eval[:,0]==k];syn=x_synthetic[y_synthetic[:,0]==k]
    ax.hist(real.ravel(),bins=30,density=True,histtype='step',label='Опорные новые')
    ax.hist(syn.ravel(),bins=30,density=True,histtype='step',label='cGAN')
    ax.set_title(f'Класс {k}');ax.set_xlabel('x');ax.set_ylabel('Плотность');ax.legend();ax.grid()
plt.tight_layout();plt.show()
print('Выбрано условий по классам:',np.bincount(y_synthetic.ravel()))

# По желанию: экспорт после проверки качества, затем скачать CSV через интерфейс Jupyter.
EXPORT_CSV=False
if EXPORT_CSV:
    np.savetxt('gan4_synthetic_balanced.csv',np.column_stack([x_synthetic,y_synthetic]),
               delimiter=',',header='x,condition_y',comments='')


**Ответ D.** Зачем y нужен обеим сетям? … . Как получить только миноритарный класс? … .
Одинаковое количество заданных меток гарантирует ли правильность x внутри каждого класса? … .
Почему для всей сбалансированной выборки нельзя требовать совпадения с исходной смесью 90/10? … .
Чем наш опыт отличается от SDG-GAN и CTGAN? … .


## E. Качество синтетики и коллапс режимов

**E1.** Допишите среднюю абсолютную разность квантилей двух одномерных выборок.
Это простая дискретная аппроксимация расстояния по квантилям, не универсальный сертификат качества.
Вычисляем также среднее, std и долю x>0 (прокси правого режима, а не предсказанная метка).
**E2.** Сравните реальный новый набор, GAN и намеренно вырожденный генератор,
который всегда возвращает −1.5. Затем отдельно оцените оба класса cGAN.

Коллапс означает недостаточный охват разнообразия/режимов; точное копирование строк не обязательно.
Один и тот же x у разных z — крайний наглядный случай. Сходство распределений не доказывает
полезность для конечной задачи и не дает гарантий конфиденциальности.


In [ ]:
def quantile_gap(a,b):
    q=np.linspace(.01,.99,99)
    # TODO: среднее абсолютное различие квантилей a и b
    raise NotImplementedError('Допишите quantile_gap')

assert np.isclose(quantile_gap([0,1,2],[0,1,2]),0)
assert np.isclose(quantile_gap([0,1,2],[2,3,4]),2)
collapsed=np.full_like(fake_unconditional,-1.5)
def describe(a,ref):
    a=np.asarray(a).ravel()
    return dict(mean=float(a.mean()),std=float(a.std()),right_fraction=float(np.mean(a>0)),qgap=float(quantile_gap(a,ref)))
for name,data in [('GAN',fake_unconditional),('Вырожденная модель',collapsed)]:
    print(name,describe(data,x_eval))
for k in [0,1]:
    print('cGAN, класс',k,describe(x_synthetic[y_synthetic[:,0]==k],x_eval[y_eval[:,0]==k]))

fig,ax=plt.subplots(figsize=(7,3))
for data,label in [(x_eval,'Опорное 90/10'),(fake_unconditional,'GAN'),(collapsed,'Вырожденная модель')]:
    ax.hist(data.ravel(),bins=60,range=(-3,3),density=True,histtype='step',label=label)
ax.set_xlabel('x');ax.set_ylabel('Плотность');ax.legend();ax.grid();plt.show()


**Ответ E.** Запишите значения показателей и два ограничения генерации: … .
Чем коллапс отличается от простого «запоминания»? … .
Как проверить пользу синтетики для реального классификатора? … .
Почему улучшение из прикладного исследования нельзя обещать для любого датасета? … .


## F. Как выбрать семейство GAN

Карточки для обсуждения. Читайте вместе с указанными слайдами. Полные модели ниже не обучаем.

**Таблицы.** CTGAN учитывает смешанные числовые/категориальные
столбцы, многомодальность и редкие категории. В описании лекции G и D имеют по два
полносвязных скрытых слоя; G использует BatchNorm/ReLU, tanh для числового представления
и Gumbel-Softmax для индикаторов режима/категорий; D — LeakyReLU и dropout.
ReLU — кусочно-линейная нелинейная активация. Выбор условий при обучении помогает
представить редкие категории; это не обещание равномерного распределения всех числовых данных.
Полноценная CTGAN также требует специального представления непрерывных колонок.
DECAF вводит причинный граф и условные механизмы для переменных: генерация идет от корней
к потомкам в топологическом порядке. Изменение допустимых причинных путей при выводе
используют для устранения нежелательного смещения по выбранному критерию fairness.
Fairness не тождественна истинности данных; причинные предположения нужно обосновывать.

**Изображения.** DCGAN использует сверточные сети: strided
convolution в D, обучение повышению разрешения в G, BatchNorm (на слайде 38 опечатка
«бинарная нормализация»); без глубокого полносвязного скрытого стека. G: ReLU внутри,
tanh на выходе; D: LeakyReLU внутри, выход для real/fake отдельно. Это рекомендации
конкретной DCGAN, а не обязательная архитектура любой GAN.
StyleGAN из лекции (первая версия): отображение z в промежуточное W, управление стилем
через AdaIN и отдельный шум в слоях синтеза. На слайде 48 у приведенного генератора
26.2 млн параметров, mapping — 8 слоев, synthesis — 18; числа не универсальны для всех StyleGAN.

**Домены.** CycleGAN обучает G:X→Y и F:Y→X с двумя
дискриминаторами по непарным наборам. Циклическая потеря требует F(G(x))≈x и G(F(y))≈y;
ее считают по восстановлению, а дискриминаторы проверяют принадлежность нужному домену.
Парные соответствия для обучения не обязательны. Лингвистическая аналогия — перевод туда и обратно.
SIDGAN — применение идеи синтеза пар для обучения обработки видео при слабом освещении;
это не утверждение, что любой смартфон запускает CycleGAN при съемке.

**Приватность и неопределенность.** PATE-GAN использует
учителей на непересекающихся частях приватных данных и приватную агрегацию ответов для
обучения student-дискриминатора. Для гарантий нужны механизм шума и учет бюджета приватности;
одного обычного большинства учителей недостаточно. DP ограничивает влияние отдельной записи
при заданных параметрах, а не обещает абсолютное нераскрытие при любых обстоятельствах.
BayesGAN рассматривает распределения параметров G/D и выборки из них; это помогает
учитывать неопределенность и разнообразие. Это не универсальная защита от коллапса и не
автоматическая дифференциальная приватность.

**Кейсы лекции: что нужно знать для В11, В18, В24, В36–37, В40, В43–44.**

| Слайды | Описанный кейс / результат в источнике | Как трактовать |
|---|---|---|
| 15–16 | Позиционирование: 10% реальных +90% синтетических дали близкий результат к полным данным | Результат конкретного опыта, не правило «можно всегда убрать 90% данных» |
| 21–23 | SDG-GAN и мошенничество: указано улучшение на 5% | Требует уточнения исходной метрики и протокола; в нашем коде не воспроизводится |
| 29–30 | CTGAN и сетевые вторжения: указано повышение точности на 8% | Не общее свойство CTGAN; процентные пункты и относительные проценты нужно различать |
| 39 | DCGAN, КТ печени: 78.6% →85.7% | Разница 7.1 процентного пункта |
| 43–44 | Ансамбль DCGAN, МРТ метастазов: на слайде написано «увеличить ложноположительные на 4.5%» при 90% чувствительности | Увеличение FP не улучшение при фиксированной чувствительности; направление эффекта требует проверки первоисточника |
| 50–51 | StyleGAN для дополнения рентгеновских данных при классификации COVID-пневмонии | Пример применения; точного численного эффекта на этих слайдах нет |
| 58–59 | CycleGAN для приближения рендеров самолетов к реальным снимкам | Перенос домена, дополнение нехватки размеченных данных |
| 60–61 | SIDGAN: динамические пары для обучения RAW-to-RGB при слабом освещении | В лекции отмечены качество цвета, артефакты и временная согласованность |

**1.** Для строки с редкой категорией и числовыми полями выберите CTGAN или
DECAF; объясните, когда нужен причинный граф. Назовите активации и способ представления категорий.

**2.** Сопоставьте DCGAN/StyleGAN/CycleGAN задачам: обычный синтез изображений;
управление стилями; непарный перенос доменов. Вычислите циклическую потерю ниже.

**3.** Различите PATE-GAN и BayesGAN и найдите две чрезмерные формулировки в кейсах/выводах лекции.


In [ ]:
def cycle_loss(x,G_xy,F_yx):
    # TODO: средняя абсолютная ошибка восстановления F(G(x)) относительно x
    raise NotImplementedError('Допишите cycle_loss')

x=np.array([-1.,0.,1.]);target_y=x+5
identity=lambda a:np.asarray(a)
value=cycle_loss(x,identity,identity)
assert np.isclose(value,0)
print('Циклическая ошибка у тождественных отображений:',value)
print('Среднее G(x):',identity(x).mean(),'; среднее целевого Y:',target_y.mean())
print('Нулевой cycle loss без состязательной проверки домена недостаточен')


**Ответ F.**

| Задача | Подход | Особенность / ограничение |
|---|---|---|
| Строки со смешанными типами и редкими категориями | … | … |
| Изменение недопустимых причинных путей | … | … |
| Синтез изображений сверточным генератором | … | … |
| Управление масштабами стиля изображения | … | … |
| Перенос между непарными доменами | … | … |
| Формальная приватность данных обучения | … | … |
| Неопределенность параметров и разнообразие | … | … |

CTGAN: активации G …; D … . DECAF: порядок синтеза … .
DCGAN: слои/нормализация …; активации … . StyleGAN: W/AdaIN/шум …; 26.2M относится к … .
Почему loss цикла равен нулю, а нужный домен не получен? … .
В чем опасность обещаний «полной конфиденциальности» и «защиты от коллапса» без условий? … .
Выберите один кейс из таблицы: задача, результат по лекции и ограничение вывода … .


## G. Итоговый контроль
Пять ответов индивидуально за 5 минут, затем разбор 2 минуты.

1. Что получают и выдают G/D? Какие параметры меняются на каждом из двух шагов?
2. Напишите L_D и выбранный L_G; объясните, почему градиент проходит через замороженную D.
3. Как получить 1000 строк редкого класса с cGAN? Почему баланс меток не доказывает качество строк?
4. Назовите две диагностики разнообразия. Почему D≈0.5 не является достаточным критерием?
5. Выберите модель для непарного переноса изображений и модель с механизмом DP;
   объясните, почему обычная GAN не гарантирует конфиденциальность.

**Ответы 1–5:** …

**Итог (2–3 предложения).** Что научилась генерировать модель, какое ограничение обнаружено
и что еще проверять перед применением синтетических данных? …

Сдать заполненный notebook. Опционально приложить CSV, если включен экспорт.


## Дополнительно
- Повторить опыт с другим seed и отношением шагов D:G; не подбирать их по финальному тесту.
- Сделать контрольный опыт с равномерным real-сэмплингом, но без условия y, чтобы отделить эффекты.
- Проверить полезность синтетики на отдельном реальном тесте downstream-модели.
- Реализовать feature matching или перейти к двумерным данным.
- Изучить полноценные CTGAN/DCGAN в отдельной среде; это другие вычислительные требования.

## Первоисточники для уточнения лекции
- Goodfellow et al., GAN: https://arxiv.org/abs/1406.2661
- Mirza, Osindero, cGAN: https://arxiv.org/abs/1411.1784
- Radford et al., DCGAN: https://arxiv.org/abs/1511.06434
- Xu et al., CTGAN: https://arxiv.org/abs/1907.00503
- Karras et al., StyleGAN: https://arxiv.org/abs/1812.04948
- Zhu et al., CycleGAN: https://arxiv.org/abs/1703.10593
- Jordon et al., PATE-GAN: https://openreview.net/pdf?id=S1zk9iRqF7
- Saatchi, Wilson, Bayesian GAN: https://arxiv.org/abs/1705.09558
- DECAF: https://arxiv.org/abs/2110.12884
